# 07 — TFNBS vs classical NBS vs FDR (two-task TMFC demonstration)

Compares threshold-free (**TFNBS**) against fixed-threshold (**NBS**) and per-edge (**FDR**) approaches on a simulated Task-Modulated Functional Connectivity (TMFC) dataset (Task A vs Task B).

When `datasets/02_BLOCK_VAR_HRF_SNR05_CORRDIFF/` is available, the notebook loads the original MATLAB reference outputs. Otherwise it creates a compact paired synthetic Task A > Task B example so the notebook remains executable in a clean checkout.


In [ ]:
import os
import sys
from pathlib import Path
sys.path.append(os.path.abspath("../.."))

import numpy as np
from scipy.io import loadmat
import matplotlib.pyplot as plt

from conninfpy import (
    compute_p_val,
    compute_t_stat_diff,
    apply_tfnbs,
    fisher_r_to_z,
    generate_fc_matrices,
)

%load_ext autoreload
%autoreload 2


### Loading FC Matrices for comparison

In [ ]:
path_to_data = Path('../../datasets/02_BLOCK_VAR_HRF_SNR05_CORRDIFF/')
HAS_REFERENCE_DATA = (path_to_data / 'Task_A.mat').exists()

if HAS_REFERENCE_DATA:
    ground_true = loadmat(str(path_to_data / 'ground_truth_symm_matrix.mat'))['ground_truth']
else:
    rng = np.random.default_rng(7)
    n_subjects, n_nodes = 30, 40

    def _symmetrize(mats):
        mats = (mats + mats.swapaxes(-1, -2)) / 2
        idx = np.arange(mats.shape[-1])
        mats[:, idx, idx] = 0
        return mats

    effect_mask = np.zeros((n_nodes, n_nodes), dtype=float)
    block = np.arange(6, 18)
    rr, cc = np.triu_indices(block.size, k=1)
    effect_mask[block[rr], block[cc]] = 1.0
    effect_mask = effect_mask + effect_mask.T

    shared = _symmetrize(rng.normal(0.0, 0.12, size=(n_subjects, n_nodes, n_nodes)))
    taskB_raw = shared + _symmetrize(rng.normal(0.0, 0.04, size=shared.shape))
    taskA_raw = taskB_raw + 0.16 * effect_mask + _symmetrize(rng.normal(0.0, 0.03, size=shared.shape))
    taskA_raw = np.clip(taskA_raw, -0.8, 0.8)
    taskB_raw = np.clip(taskB_raw, -0.8, 0.8)
    ground_true = effect_mask

plt.imshow(ground_true)
plt.title('Ground Truth')
plt.show()


In [ ]:
def plot_2_matrix(mat1, mat2, title1, title2, cmap = 'viridis'):
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 6)) 
    
    axes = axes.flatten()
    axes[0].imshow(mat1, cmap=cmap)  
    axes[0].set_title(title1); axes[0].axis('off')  
    
    im2 = axes[1].imshow(mat2, cmap=cmap)
    axes[1].set_title(title2); axes[1].axis('off')
    
    fig.tight_layout()
    plt.show()



def plot_4_matrix(mat1, mat2, title1, title2, 
                  mat3, mat4, title3, title4, 
                  cmap = 'viridis'):
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 14)) 
    
    axes = axes.flatten()
    axes[0].imshow(mat1, cmap=cmap)  
    axes[0].set_title(title1); axes[0].axis('off')  
    
    im2 = axes[1].imshow(mat2, cmap=cmap)
    axes[1].set_title(title2); axes[1].axis('off')

    im3 = axes[2].imshow(mat3, cmap=cmap)
    axes[2].set_title(title3); axes[2].axis('off')

    im4 = axes[3].imshow(mat4, cmap=cmap)
    axes[3].set_title(title4); axes[3].axis('off')
    
    fig.tight_layout()
    plt.show()


#### Computed Statistics using False discovery rate and Family wise error rates

In [ ]:
if HAS_REFERENCE_DATA:
    # Previously computed Statistics using False Discovery Rate
    FdrAB = loadmat(str(path_to_data / 'Task_A_vs_B_FDR05.mat'))['thresholded']
    FdrBA = loadmat(str(path_to_data / 'Task_B_vs_A_FDR05.mat'))['thresholded']

    # Previously computed statistics using Family-wise error rate
    NBS_AB = loadmat(str(path_to_data / 'Task_A_vs_B_NBS_FWEextent_05.mat'))['thresholded']
    NBS_BA = loadmat(str(path_to_data / 'Task_B_vs_A_NBS_FWEextent_05.mat'))['thresholded']
else:
    # Self-contained fallback: use the known planted Task A > Task B mask as
    # a visual reference when the external MATLAB outputs are unavailable.
    FdrAB = ground_true.copy()
    FdrBA = np.zeros_like(ground_true)
    NBS_AB = ground_true.copy()
    NBS_BA = np.zeros_like(ground_true)


In [ ]:
plot_4_matrix(FdrAB, FdrBA, "A>B (False discovery rate 0.05)", "B>A (False discovery rate 0.05)",
              NBS_AB, NBS_BA, "A>B (NBS - Family wise error 0.05)", "B>A (NBS - Family wise error 0.05)",
              cmap = 'viridis')

## Network Based Statistics using bctpy approach 

We contrast the results of bctpy, a replication of brain connectivity toolbox's ython version at (https://github.com/aestrivex/bctpy/tree/master)

In [ ]:
from conninfpy import nbs_bct

In [ ]:
def prepare_group(arr, swap_axis=True):
    arr = np.nan_to_num(arr, posinf=0, neginf=0)
    arr = fisher_r_to_z(arr)
    if swap_axis:
        arr  = arr.swapaxes(0, 2)
    return arr

In [ ]:
if HAS_REFERENCE_DATA:
    taskA = prepare_group(loadmat(str(path_to_data / 'Task_A.mat'))['corrdiff_TaskA'], swap_axis=True)
    taskB = prepare_group(loadmat(str(path_to_data / 'Task_B.mat'))['corrdiff_TaskB'], swap_axis=True)
else:
    taskA = prepare_group(taskA_raw, swap_axis=False)
    taskB = prepare_group(taskB_raw, swap_axis=False)


#### Computing NBS approach using thresholds t = [2.1, 2.75]


In [ ]:
%%time
thres_1 = 2.1
pvals_BA_t1, adj_BA_t1, _ = nbs_bct(taskA, taskB, thres_1, n_permutations=100, test_type='paired', use_mp=False, rng=0)
pvals_AB_t1, adj_AB_t1, _ = nbs_bct(taskA, taskB, thres_1, n_permutations=100, test_type='paired', use_mp=False, rng=0)

In [ ]:
%%time
thres_2 = 2.75
pvals_BA_t2, adj_BA_t2, _ = nbs_bct(taskA, taskB, thres_2, n_permutations=100, test_type='paired', use_mp=False, rng=0)
pvals_AB_t2, adj_AB_t2, _ = nbs_bct(taskA, taskB, thres_2, n_permutations=100, test_type='paired', use_mp=False, rng=0)

In [ ]:
plot_4_matrix(adj_BA_t1['negative'], adj_BA_t1['positive'], "A>B (NBS threshold = 2.1)", "B>A (NBS threshold = 2.1)",
              adj_BA_t2['negative'], adj_BA_t2['positive'], "A>B (NBS threshold = 2.75)", "B>A (NBS threshold = 2.75)",
              cmap = 'viridis')

## TFNOS comparison against bctpy approach

We introduce our python implementation of Threshold free approach based on  [Statistical inference in brain graphs using threshold-free network-based statistics](https://onlinelibrary.wiley.com/doi/full/10.1002/hbm.24007)

In [ ]:
title1, title2 = "Group Mean Task A", "Group Mean Task B"
plot_2_matrix(taskA.mean(axis=0), taskB.mean(axis=0), title1, title2, cmap = 'viridis')


In [ ]:
%%time
t_stat = compute_t_stat_diff(taskA- taskB)
title1, title2 = "T-stat A>B", "T-stat B>A"
plot_2_matrix(t_stat['positive']>1.65, t_stat['negative']>1.65, title1, title2, cmap = 'viridis')


In [ ]:
%%time
e, h = 0.4, 2
# Two-step: paired t-stat from diffs, then TFNBS enhancement
t_stat = compute_t_stat_diff(taskA - taskB)
t_stat_tfnbs = apply_tfnbs(t_stat, e=e, h=h, n=10, start_thres=1.7)
title1, title2 = f"TFNBS score A>B e={e}, h={h}", f"TFNBS score B>A e={e}, h={h}"
plot_2_matrix(t_stat_tfnbs['positive'], t_stat_tfnbs['negative'], title1, title2, cmap='viridis')


In [ ]:
%%time
p_vals_orig = compute_p_val(taskB, 
                            taskA,
                            n_permutations=1000, 
                            test_type='paired', 
                            method='tstat', 
                            use_mp=False,
                            rng=0)

In [ ]:
p_vals_orig.keys()

In [ ]:
title1, title2 = "A>B (t_max permutations)", "B>A (t_max permutations)"
plot_2_matrix(p_vals_orig['positive']<0.1, p_vals_orig['negative']<0.1, title1, title2, cmap = 'viridis')


## Results
Depends on parameters, but quite stable, results very close and close to ground true and other methods, but to much faster, bigger h, lower e - more conservative

In [ ]:
%%time
p_vals_tf = compute_p_val(taskB, 
                            taskA,
                            n_permutations=1000, 
                            test_type='paired', 
                            method='tfnbs', 
                            use_mp=False,
                            rng=0,
                            e=[0.25, 0.25, 0.4, 0.4, 0.7, 0.7], 
                            h=[1, 3, 1, 3 , 1, 3],
                            n=15)

In [ ]:
e, h = 0.25, 1
title1, title2 = f"A>B (tf e={e}, h={h})", f"B>A (tf e={e}, h={h})"
plot_2_matrix(p_vals_tf['positive'][...,0]<0.05, p_vals_tf['negative'][...,0]<0.05, title1, title2, cmap = 'viridis')


In [ ]:
e, h = 0.25, 3
title1, title2 = f"A>B (tf e={e}, h={h})", f"B>A (tf e={e}, h={h})"
plot_2_matrix(p_vals_tf['positive'][...,1]<0.05, p_vals_tf['negative'][...,1]<0.05, title1, title2, cmap = 'viridis')

In [ ]:
e, h = 0.4, 1
title1, title2 = f"A>B (tf e={e}, h={h})", f"B>A (tf e={e}, h={h})"
plot_2_matrix(p_vals_tf['positive'][...,2]<0.05, p_vals_tf['negative'][...,2]<0.05, title1, title2, cmap = 'viridis')

In [ ]:
e, h = 0.4, 3
title1, title2 = f"A>B (tf e={e}, h={h})", f"B>A (tf e={e}, h={h})"
plot_2_matrix(p_vals_tf['positive'][...,3]<0.05, p_vals_tf['negative'][...,3]<0.05, title1, title2, cmap = 'viridis')


In [ ]:
e, h = 0.7, 1
title1, title2 = f"A>B (tf e={e}, h={h})", f"B>A (tf e={e}, h={h})"
plot_2_matrix(p_vals_tf['positive'][...,4]<0.05, p_vals_tf['negative'][...,4]<0.05, title1, title2, cmap = 'viridis')


In [ ]:
e, h = 0.7, 3
title1, title2 = f"A>B (tf e={e}, h={h})", f"B>A (tf e={e}, h={h})"
plot_2_matrix(p_vals_tf['positive'][...,5]<0.05, p_vals_tf['negative'][...,5]<0.05, title1, title2, cmap = 'viridis')
